In [2]:
pip install selenium


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install webdriver-manager


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install bs4


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [28]:
#IMPORTACIONES
from selenium import webdriver
from selenium.webdriver.common.by import By
import time
import random
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
import csv
import requests
import pandas as pd
import re
import os

from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

In [23]:
#WEBDRIVER
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Modo headless
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920x1080")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

In [ ]:
# HIOJA DE DATOS DE LOS VINOS
# Leer los enlaces desde un archivo de texto
with open("/Users/josetudela/Projects/vinos_grupo2/Proyecto_grupo2_vinos/archivos/Vinos blancos de 9 a 11.txt", 'r', encoding='utf-8-sig') as f:
    urls = f.readlines()  # Lee todos los enlaces en el archivo



# Limitar a los primeros 5 enlaces
#urls = urls[:2]

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Inicializar una lista para guardar todos los datos
all_wine_data = []

# Recorrer cada URL en la lista
for url in urls:
    url = url.strip()  # Eliminar cualquier espacio en blanco o salto de línea

    try:
        # Solicitud a la página
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Inicializar datos
        wine_data = {}

        # País, región, bodega, tipo de vino, uva
        breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
        country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
        region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
        winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
        wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'
        #grape = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-grape'}).text.strip() if breadcrumbs else 'No disponible'


        #Grape
        grapes = soup.find_all("a", class_="anchor_anchor__m8Qi- wineFacts__link--3aTg9")
        grape = [grape.text.strip() for grape in grapes if "grapes" in grape["href"]]
        grape = ', '.join(grape) if grape else 'No disponible'


        # Nombre del vino 
        wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
        if wine_headline:
            # Tomamos todo el texto dentro del bloque, sin intentar separarlo
            name = wine_headline.get_text(strip=True)
        else:
            name = 'No disponible'
        
            # Si el nombre del vino contiene el nombre de la bodega, eliminamos la bodega del nombre
        if winery.lower() in name.lower():
            name = name.replace(winery, '').strip()


        # Año
        button_elements = soup.find_all('button', class_='MuiButtonBase-root')
        year = 'No disponible'

            # Buscar en los botones primero
        for button in button_elements:
            if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
                year = button.get('aria-label').strip()
                break

            # Si no se encuentra en los botones, buscar en el span con la clase 'vivino-mui-14ngluw-componentChildren'
        if year == 'No disponible':
            year_element = soup.find('span', class_='vivino-mui-14ngluw-componentChildren')
            if year_element and year_element.text.strip().isdigit() and len(year_element.text.strip()) == 4:
                year = year_element.text.strip()

            # Si aún no se ha encontrado, buscar todos los 'span' con la clase 'vintageListRow__year--34Tuc' y tomar el primero
        if year == 'No disponible':
            vintage_section = soup.find('div', id='vintageListSection')
            if vintage_section:
                # Buscar todos los 'span' dentro del div y filtrar aquellos que contienen un año (4 dígitos)
                year_elements = vintage_section.find_all('span', string=re.compile(r'\d{4}'))
                if year_elements:
                    year = year_elements[0].string.strip()  # Usamos .string para obtener solo el texto


        # Precio
        price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
        if price_element:
            price = price_element.text.replace('€', '').replace('\xa0', '').strip()
        else:
            # Si no lo encuentra, busca el precio en la segunda clase
            price_element = soup.find(class_='purchaseAvailabilityPPC__amount--2_4GT')
            if price_element:
                # Utilizamos regex para encontrar el precio con la coma
                match = re.search(r'\d{1,3}(?:,\d{3})*(?:\.\d+)?', price_element.text)
                if match:
                    price = match.group(0).replace('\xa0', '').strip()
                else:
                    price = 'No disponible'
            else:
                price = 'No disponible'

        
        # Grados de Alcohol
        alcohol_element = soup.find(class_='wineFacts__wineFacts--2Ih8B')
            # Buscar todos los spans dentro de la tabla
        if alcohol_element:
            spans = alcohol_element.find_all('span')
            # Filtrar los spans que contienen un número seguido de '%' (grado de alcohol)
        alcohol = 'No disponible'
        for span in spans:
                # Usamos una expresión regular para buscar un número seguido de '%'
                match = re.search(r'\d+%', span.text.strip())
                if match:
                    alcohol = match.group(0)  # El valor que coincide con la expresión regular
                    alcohol = alcohol.replace('%', '')
                    break  # Detener la búsqueda cuando encontramos el primer grado de alcohol
        else:
            alcohol = 'No disponible'



        # Notas de sabor
        taste_containers = soup.find_all(class_='slider__viewPort--30MrB')
        taste_notes = []
        for container in taste_containers:
            # Encontrar todos los elementos con la clase 'tasteNote__popularKeywords--1gIa2'
            taste_keywords = container.find_all(class_='tasteNote__popularKeywords--1gIa2')

            # Recorrer cada uno de los elementos encontrados y extraer el texto
            for keyword in taste_keywords:
                if keyword.text.strip():  # Solo si no está vacío
                    taste_notes.append(keyword.text.strip())

        # Unir todas las notas en una sola cadena, separada por coma
        taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

        
        # Valoración
        rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
        rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'  # Reemplaza la coma por un punto

        # Maridajes
        food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
        pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]


        # Guardar los datos de esta URL
        wine_data = {
            'Url': url,
            'ID': re.search(r'\/(\d+)$', url).group(1),
            'Nombre': name,
            'Año': year,
            'País': country,
            'Región': region,
            'Bodega': winery,
            'Tipo de vino': wine_type,
            'Uva': grape,
            'Precio': price,
            'Valoración': rating,
            'Contenido de alcohol': alcohol,
            'Maridajes':', '.join(pairings),
            
        }
        all_wine_data.append(wine_data)
        
       
    except Exception as e:
        print(f"Error al procesar la URL {url}: {e}")

# Guardar los resultados en un archivo CSV
with open('wine_data.csv', mode='w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
    writer.writeheader()
    writer.writerows(all_wine_data)

# Convertir los datos a un DataFrame y mostrar los primeros registros
df = pd.DataFrame(all_wine_data)
pd.set_option('display.max_colwidth', None)  # No limita el ancho de las celdas
pd.set_option('display.max_rows', None)  # Sin límite de filas
pd.set_option('display.max_columns', None)  # Sin límite de columnas

print(df.head())

Error al procesar la URL : Invalid URL '': No scheme supplied. Perhaps you meant https://?
                                                                                             Url  \
0  https://www.vivino.com/ES/es/schloss-muhlenhof-spatburgunder-blanc-de-noirs-trocken/w/1420124   
1         https://www.vivino.com/ES/es/felix-lorenzo-cachazo-carrasvinas-rueda-verdejo/w/1784490   
2                                       https://www.vivino.com/ES/es/cartuxa-ea-branco/w/1712414   
3                         https://www.vivino.com/ES/es/vina-sanzo-vinas-viejas-verdejo/w/1256164   
4                                         https://www.vivino.com/ES/es/mariluna-blanco/w/1920280   

        ID                                Nombre            Año      País  \
0  1420124  Spätburgunder Blanc de Noirs Trocken  No disponible  Alemania   
1  1784490             Carrasviñas Rueda Verdejo           2024    España   
2  1712414                             EA Branco  No disponible  Portugal   


In [31]:
import os
import re
import csv
import time
import requests
from bs4 import BeautifulSoup

# Archivo de entrada
input_file = "/Users/josetudela/Projects/vinos_grupo2/Proyecto_grupo2_vinos/vinos_convers/tintos_1_15000.txt"

# Archivo de salida
output_file = "wine_data.csv"

# Archivo de punto de control (checkpoint)
checkpoint_file = "last_processed_batch.txt"

# Headers para evitar bloqueos
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Leer todas las URLs del archivo de texto
with open(input_file, 'r', encoding='utf-8-sig') as f:
    urls = [line.strip() for line in f.readlines() if line.strip().startswith("http")]

# Dividir la lista en lotes de 100
batch_size = 100
num_batches = (len(urls) // batch_size) + (1 if len(urls) % batch_size > 0 else 0)

# Verificar el último lote procesado desde el archivo de punto de control
last_processed_batch = 0
if os.path.exists(checkpoint_file):
    with open(checkpoint_file, 'r') as f:
        last_processed_batch = int(f.read().strip())

print(f"🔹 Total de URLs: {len(urls)}")
print(f"🔹 Procesando en {num_batches} lotes de {batch_size} URLs cada uno.")
print(f"🔹 Comenzando desde el lote {last_processed_batch + 1}...")

# Procesar cada batch desde el último lote procesado
for batch_index in range(last_processed_batch, num_batches):
    batch_urls = urls[batch_index * batch_size: (batch_index + 1) * batch_size]
    all_wine_data = []
    
    print(f"\n🚀 Procesando lote {batch_index + 1} de {num_batches}...")

    for url in batch_urls:
        print(f"🔄 Procesando URL: {url}")  # Mostrar la URL que se está procesando
        
        try:
            response = requests.get(url, headers=headers)
            soup = BeautifulSoup(response.content, 'html.parser')

            # Datos básicos
            wine_data = {}
            breadcrumbs = soup.find(class_='breadCrumbs__breadCrumbs--2pkcX')
            country = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-country'}).text.strip() if breadcrumbs else 'No disponible'
            region = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-region'}).text.strip() if breadcrumbs else 'No disponible'
            winery = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winery'}).text.strip() if breadcrumbs else 'No disponible'
            wine_type = breadcrumbs.find(attrs={'data-cy': 'breadcrumb-winetype'}).text.strip() if breadcrumbs else 'No disponible'

            # Uva
            grapes = soup.find_all("a", class_="anchor_anchor__m8Qi- wineFacts__link--3aTg9")
            grape = [g.text.strip() for g in grapes if "grapes" in g["href"]]
            grape = ', '.join(grape) if grape else 'No disponible'

            # Nombre del vino
            wine_headline = soup.find('div', class_='wineHeadline-module__wineHeadline--32Ety')
            name = wine_headline.get_text(strip=True) if wine_headline else 'No disponible'
            if winery.lower() in name.lower():
                name = name.replace(winery, '').strip()

            # Año
            year = 'No disponible'
            button_elements = soup.find_all('button', class_='MuiButtonBase-root')
            for button in button_elements:
                if button.get('aria-label') and button.get('aria-label').isdigit() and len(button.get('aria-label').strip()) == 4:
                    year = button.get('aria-label').strip()
                    break

            # Precio
            price_element = soup.find(class_='purchaseAvailability__currentPrice--3mO4u')
            if price_element:
                price = price_element.text.replace('€', '').replace('\xa0', '').strip()
            else:
                price_element = soup.find(class_='purchaseAvailabilityPPC__amount--2_4GT')
                match = re.search(r'\d{1,3}(?:,\d{3})*(?:\.\d+)?', price_element.text) if price_element else None
                price = match.group(0).replace('\xa0', '').strip() if match else 'No disponible'

            # Grados de Alcohol
            alcohol = 'No disponible'
            alcohol_element = soup.find(class_='wineFacts__wineFacts--2Ih8B')
            if alcohol_element:
                spans = alcohol_element.find_all('span')
                for span in spans:
                    match = re.search(r'\d+%', span.text.strip())
                    if match:
                        alcohol = match.group(0).replace('%', '')
                        break

            # Notas de sabor
            taste_containers = soup.find_all(class_='slider__viewPort--30MrB')
            taste_notes = []
            for container in taste_containers:
                taste_keywords = container.find_all(class_='tasteNote__popularKeywords--1gIa2')
                taste_notes.extend([keyword.text.strip() for keyword in taste_keywords if keyword.text.strip()])
            taste_notes = ', '.join(taste_notes) if taste_notes else 'No disponible'

            # Valoración
            rating_element = soup.find(class_='vivinoRating_averageValue__uDdPM')
            rating = rating_element.text.strip().replace(',', '.') if rating_element else 'No disponible'

            # Maridajes
            food_pairings = soup.find_all(class_='foodPairing__foodImage--2OYHg')
            pairings = [fp['aria-label'] for fp in food_pairings if fp.has_attr('aria-label')]

            # Guardar los datos
            wine_data = {
                'Url': url,
                'ID': re.search(r'\/(\d+)$', url).group(1) if re.search(r'\/(\d+)$', url) else '0',
                'Nombre': name,
                'Año': year,
                'País': country,
                'Región': region,
                'Bodega': winery,
                'Tipo de vino': wine_type,
                'Uva': grape,
                'Precio': price,
                'Valoración': rating,
                'Contenido de alcohol': alcohol,
                'Maridajes': ', '.join(pairings),
            }
            all_wine_data.append(wine_data)

        except Exception as e:
            print(f"❌ Error al procesar la URL {url}: {e}")

    # Guardar en un archivo CSV único para cada batch
    batch_output_file = f"wine_data_batch_{batch_index + 1}.csv"
    file_exists = os.path.exists(batch_output_file)

    with open(batch_output_file, mode='a', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=all_wine_data[0].keys())
        if not file_exists:
            writer.writeheader()  # Escribir los encabezados solo la primera vez
        writer.writerows(all_wine_data)

    print(f"✅ Lote {batch_index + 1} guardado en {batch_output_file}")

    # Actualizar el archivo de punto de control con el índice del lote procesado
    with open(checkpoint_file, 'w') as f:
        f.write(str(batch_index))

    # Pequeña pausa entre lotes para evitar bloqueos
    time.sleep(5)

print("\n🎯 ¡Todos los lotes han sido procesados y guardados correctamente! 🚀")

🔹 Total de URLs: 15200
🔹 Procesando en 152 lotes de 100 URLs cada uno.
🔹 Comenzando desde el lote 1...

🚀 Procesando lote 1 de 152...
🔄 Procesando URL: https://www.vivino.com/ES/es/asenjo-and-manso-silvanus-edicion-limitada-ribera-del-duero/w/3741348
🔄 Procesando URL: https://www.vivino.com/ES/es/oller-del-mas-parcel-la-margenat-especial-picapoll-negre/w/10646264
🔄 Procesando URL: https://www.vivino.com/ES/es/jimenez-landi-ataulfos/w/1375903
🔄 Procesando URL: https://www.vivino.com/ES/es/broccardo-barolo-tre-pais/w/1641117
🔄 Procesando URL: https://www.vivino.com/ES/es/jose-gil-camino-de-ribas-parcela-la-concova/w/10610046
🔄 Procesando URL: https://www.vivino.com/ES/es/kevin-bouillet-pepin-rouge/w/9177842
🔄 Procesando URL: https://www.vivino.com/ES/es/proelio-la-canal-del-rojo-garnacha/w/9181296
🔄 Procesando URL: https://www.vivino.com/ES/es/pradorey-finca-real-sitio-de-ventosilla-gran-reserva-ribera-del-duero/w/1146995
🔄 Procesando URL: https://www.vivino.com/ES/es/domaine-chanson-bea

KeyboardInterrupt: 

In [ ]:
#CARACTERISTICAS TINTOS 1
 
# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/domaine-raymond-usseglio-and-fils-la-genese/w/9716238"

# Navegar a la página
driver.get(url)

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            
            left_value = re.search(r'left:\s*(\d+)%', style)
            left_zero = int(left_value.group(1)) if left_value else 0  # Si no se encuentra, es 0


            
            if left_value:
                # Convertir a float y convertirlo en una nota del 1 al 10 (dividiendo entre 10)
                value = round(float(left_value) / 10, 1)
                # Asignar la etiqueta correspondiente
                progress_values[labels[i]] = value
                print(f"{labels[i]}: {value}")
            else:
                # Convertir a float y convertirlo en una nota del 1 al 10 (dividiendo entre 10)
                value = 0 
                # Asignar la etiqueta correspondiente
                progress_values[labels[i]] = value
                print(f"{labels[i]}: {value}")
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")

# Nombre del archivo CSV
csv_filename = "tintos/tintos_test.csv"
# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
else:
    df = pd.DataFrame(columns=["ID"] + labels)
# Convertir ID a string para evitar errores de tipo
df["ID"] = df["ID"].astype(str)
# Verificar si la ID ya existe en el CSV

ID = url.split("/w/")[1].split("?")[0] if "/w/" in url else "Desconocido"

if ID in df["ID"].values:
    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
    print(f"Datos actualizados para el vino con ID: {ID}")
else:
    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
    print(f"Nuevo registro agregado con ID: {ID}")
# Guardar de nuevo el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()

In [ ]:
#CARACTERISTICAS TINTOS 2
 
# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/domaine-raymond-usseglio-and-fils-la-genese/w/9716238"

# Navegar a la página
driver.get(url)

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            style = element.get_attribute('style')
            left_value = re.search(r'left:\s*(\d+(\.\d+)?)%', 'style')
            left_zero = int(left_value.group(1)) if left_value else 0  # Si no se encuentra, es 0


            
            if left_value:
                # Convertir a float y convertirlo en una nota del 1 al 10 (dividiendo entre 10)
                value = round(float(left_value) / 10, 1)
                # Asignar la etiqueta correspondiente
                progress_values[labels[i]] = value
                print(f"{labels[i]}: {value}")
            else:
                # Convertir a float y convertirlo en una nota del 1 al 10 (dividiendo entre 10)
                value = 0 
                # Asignar la etiqueta correspondiente
                progress_values[labels[i]] = value
                print(f"{labels[i]}: {value}")
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")

# Nombre del archivo CSV
csv_filename = "/Users/josetudela/Projects/vinos_grupo2/Proyecto_grupo2_vinos/vinos_convers/tintos_test.csv"
# Cargar datos existentes o crear un nuevo DataFrame
if os.path.exists(csv_filename):
    df = pd.read_csv(csv_filename)
else:
    df = pd.DataFrame(columns=["ID"] + labels)
# Convertir ID a string para evitar errores de tipo
df["ID"] = df["ID"].astype(str)
# Verificar si la ID ya existe en el CSV

ID = url.split("/w/")[1].split("?")[0] if "/w/" in url else "Desconocido"

if ID in df["ID"].values:
    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
    print(f"Datos actualizados para el vino con ID: {ID}")
else:
    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
    print(f"Nuevo registro agregado con ID: {ID}")
# Guardar de nuevo el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()

Ligero/Poderoso: 0
Suave/Tánico: 0
Seco/Dulce: 0
Débil/Ácido: 0


EmptyDataError: No columns to parse from file

In [ ]:
#CARACTERISTICAS ESPUMOSOS:


# URL de la página de Vivino
url = "https://www.vivino.com/ES/es/bollinger-vieilles-vignes-francaises-blanc-de-noirs-brut-champagne/w/18938?year=2009&price_id=30860996"

# Navegar a la página
driver.get(url)

labels = ["Ligero/Poderoso", "Débil/Ácido", "Amable/Con Burbujas"]

# Diccionario para guardar los resultados
progress_values = {}

try:
    # Buscar todas las barras de progreso
    progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
    # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
    if len(progress_elements) == len(labels):
        # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
        for i, element in enumerate(progress_elements):
            left_value = element.get_attribute('style').split('left: ')[1].split('%')[0] if 'left' in element.get_attribute('style') else None
            
            if left_value:
                # Convertir a float y mapearlo con los valores predefinidos
                value = round(float(left_value) / 10, 1)
                # Asignar la etiqueta correspondiente con los valores predefinidos
                progress_values[labels[i]] = value
                print(f"{labels[i]}: {value}")
            else:
                print(f"No se encontró el atributo 'left' para {labels[i]}.")
    else:
        print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
except Exception as e:
    print(f"Error durante la extracción: {e}")

# Cerrar el navegador
driver.quit()

In [24]:
#BARRA CARACTERÍSITCAS TINTOS

# Initialize the WebDriver
#driver = webdriver.Chrome(options=options)

# Read URLs from a .txt file
with open('/Users/josetudela/Projects/vinos_grupo2/Proyecto_grupo2_vinos/vinos_convers/def_tintos1.txt', 'r') as file:
    urls_list = [line.strip() for line in file.readlines()]

urls_list = urls_list[:5]

# Mapeo de etiquetas y las clases de las barras de progreso
labels = ["Ligero/Poderoso", "Suave/Tánico", "Seco/Dulce", "Débil/Ácido"]

# Nombre del archivo CSV
csv_filename = "def_tintos.csv"
# Cargar datos existentes o crear un nuevo DataFrame
#if os.path.exists(csv_filename):
df = pd.read_csv(csv_filename)
#else:
    #df = pd.DataFrame(columns=["ID"] + labels)


# Diccionario para guardar los resultados
progress_values = {}

# Process each URL in the list
for original_url in urls_list:
    print(f"Procesando URL: {original_url}")  # Show progress
    match = re.search(r'\/(\d+)$', original_url)  # Buscar el ID en la URL
    ID = match.group(1) if match else "Desconocido"  # Si no se encuentra, asignar "Desconocido"

    try:
        # Open the URL
        driver.get(original_url)

        # Wait for the page to load completely (adjust the condition as needed)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
        #time.sleep(3)

        # Buscar todas las barras de progreso
        progress_elements = driver.find_elements(By.CLASS_NAME, 'indicatorBar__progress--3aXLX')
    
        # Asegurarnos de que tenemos el mismo número de barras de progreso que etiquetas
        if len(progress_elements) == len(labels):
            # Iterar sobre las barras de progreso y asignarles las etiquetas correspondientes
            for i, element in enumerate(progress_elements):
                style = element.get_attribute('style')
                print(f"Barra {i+1}: {style}")  # Imprimir el atributo style para depuración
                # Sacar el valor del left :
                valor_left = re.search(r'left:\s*([0-9.]+)', style).group(1)
                valor_left = round(float(valor_left)/10,1)
                progress_values[labels[i]] = valor_left
                
    
                # Convertir ID a string para evitar errores de tipo
                df["ID"] = df["ID"].astype(str)
                if ID in df["ID"].values:
                    df.loc[df["ID"] == ID, labels] = [progress_values[label] for label in labels]
                    print(f"Datos actualizados para el vino con ID: {ID}")
                else:
                    df = pd.concat([df, pd.DataFrame([progress_values])], ignore_index=True)
                    print(f"Nuevo registro agregado con ID: {ID}")

        else:
            print("El número de barras de progreso no coincide con el número de etiquetas esperadas.")
    
    except Exception as e:
        print(f"Error durante la extracción: {e}")

    # Guardar de nuevo el archivo CSV
df.to_csv(csv_filename, index=False)
print(f"Datos guardados en {csv_filename}")

# Cerrar el navegador
driver.quit()

Procesando URL: https://www.vivino.com/ES/es/sine-qua-non-next-of-kyn/w/1914259
Barra 1: width: 15%; left: 71.1339%;
Nuevo registro agregado con ID: 1914259
Barra 2: width: 15%; left: 50.6634%;
Nuevo registro agregado con ID: 1914259
Barra 3: width: 15%; left: 16.2308%;
Nuevo registro agregado con ID: 1914259
Barra 4: width: 15%; left: 47.5599%;
Nuevo registro agregado con ID: 1914259
Procesando URL: https://www.vivino.com/ES/es/harlan-estate-red/w/1218210
Barra 1: width: 15%; left: 84.5056%;
Nuevo registro agregado con ID: 1218210
Barra 2: width: 15%; left: 57.2083%;
Nuevo registro agregado con ID: 1218210
Barra 3: width: 15%; left: 14.822%;
Nuevo registro agregado con ID: 1218210
Barra 4: width: 15%; left: 63.3235%;
Nuevo registro agregado con ID: 1218210
Procesando URL: https://www.vivino.com/ES/es/sine-qua-non-profuga-grenache-california/w/9375801
Barra 1: width: 15%; left: 42.8606%;
Nuevo registro agregado con ID: 9375801
Barra 2: width: 15%; left: 32.6442%;
Nuevo registro agregad

In [26]:
df.shape

(56, 5)